In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

# Diretório raiz do projeto
ROOT = Path.cwd().parent

# Diretórios do projeto
DATA_DIR = ROOT / "data"
FONTE_ORIGINAL = DATA_DIR / "fonte_original"
DERIVADOS = DATA_DIR / "derivados"
FONTES_PUBLICAS = DATA_DIR / "fontes_publicas"

print(f"Diretório do projeto: {ROOT}")
print(f"Diretório de dados: {DATA_DIR}")
print(f"Base original: {FONTE_ORIGINAL}")
print(f"Dados derivados: {DERIVADOS}")
print(f"Fontes públicas: {FONTES_PUBLICAS}")

Diretório do projeto: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo
Diretório de dados: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data
Base original: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fonte_original
Dados derivados: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\derivados
Fontes públicas: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fontes_publicas


In [3]:
# Inventário dos arquivos da pasta de origem

arquivos_originais = sorted(FONTE_ORIGINAL.iterdir())

if arquivos_originais:
    for arquivo in arquivos_originais:
        if arquivo.is_file():
            print(
                f"Arquivo: {arquivo.name} | Tamanho: {arquivo.stat().st_size:,} bytes"
            )
        elif arquivo.is_dir():
            print(f"Pasta: {arquivo.name}/")
else:
    print("A pasta fonte_original está vazia.")

Arquivo: Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv | Tamanho: 34,581 bytes


In [8]:
# ============================================================
# PROJETO: UM OLHAR DE QUEM ESTÁ NO CAMPO
# ETAPA 01 — INSPEÇÃO INICIAL DA BASE DE DADOS
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. DEFINIÇÃO DOS CAMINHOS
# ------------------------------------------------------------

# Diretório do projeto: pasta acima de notebooks
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data"
FONTE_ORIGINAL = DATA_DIR / "fonte_original"
DERIVADOS = DATA_DIR / "derivados"
FONTES_PUBLICAS = DATA_DIR / "fontes_publicas"

print("=" * 70)
print("UM OLHAR DE QUEM ESTÁ NO CAMPO")
print("ETAPA 01 — INSPEÇÃO INICIAL")
print("=" * 70)

print(f"\nDiretório do projeto: {ROOT}")
print(f"Diretório de dados: {DATA_DIR}")
print(f"Base original: {FONTE_ORIGINAL}")
print(f"Dados derivados: {DERIVADOS}")
print(f"Fontes públicas: {FONTES_PUBLICAS}")

# ------------------------------------------------------------
# 2. VERIFICAÇÃO DA ESTRUTURA DO PROJETO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. ESTRUTURA DE DIRETÓRIOS")
print("=" * 70)

for diretorio in [DATA_DIR, FONTE_ORIGINAL, DERIVADOS, FONTES_PUBLICAS]:
    status = "EXISTE" if diretorio.exists() else "NÃO ENCONTRADO"
    print(f"{status}: {diretorio}")

# ------------------------------------------------------------
# 3. INVENTÁRIO DOS ARQUIVOS ORIGINAIS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. INVENTÁRIO DA FONTE ORIGINAL")
print("=" * 70)

arquivos_csv = sorted(FONTE_ORIGINAL.glob("*.csv"))

if not arquivos_csv:
    raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {FONTE_ORIGINAL}")

for arquivo in arquivos_csv:
    tamanho_kb = arquivo.stat().st_size / 1024
    print(f"- {arquivo.name} | {tamanho_kb:.2f} KB")

# ------------------------------------------------------------
# 4. IDENTIFICAÇÃO DA BASE
# ------------------------------------------------------------

# A inspeção inicial deve trabalhar com a base baseline de 36
# respostas, sem presumir que ela representa a base consolidada.

candidatos = [
    arquivo
    for arquivo in arquivos_csv
    if "BASELINE_36_RESPOSTAS" in arquivo.name.upper()
]

if len(candidatos) != 1:
    raise ValueError(
        "Era esperado encontrar exatamente um CSV com "
        "'BASELINE_36_RESPOSTAS' no nome. "
        f"Foram encontrados {len(candidatos)}."
    )

ARQUIVO_BASE = candidatos[0]

print(f"\nArquivo selecionado: {ARQUIVO_BASE.name}")

# ------------------------------------------------------------
# 5. LEITURA SEGURA DO CSV
# ------------------------------------------------------------

# dtype=str preserva os valores como foram registrados.
# keep_default_na=False evita converter automaticamente textos
# como 'NA' em valores ausentes.

try:
    df = pd.read_csv(
        ARQUIVO_BASE, sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False
    )

except UnicodeDecodeError:
    df = pd.read_csv(
        ARQUIVO_BASE, sep=",", encoding="latin-1", dtype=str, keep_default_na=False
    )

print("\n" + "=" * 70)
print("3. DIMENSÕES DA BASE")
print("=" * 70)

print(f"Quantidade de registros: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")
print(f"Total de células: {df.size}")

# ------------------------------------------------------------
# 6. INSPEÇÃO DOS NOMES DAS COLUNAS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. NOMES DAS COLUNAS")
print("=" * 70)

for numero, coluna in enumerate(df.columns, start=1):
    print(f"{numero:02d}. {coluna}")

# ------------------------------------------------------------
# 7. VERIFICAÇÃO DE NOMES DUPLICADOS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. COLUNAS DUPLICADAS")
print("=" * 70)

colunas_duplicadas = df.columns[df.columns.duplicated()].tolist()

if colunas_duplicadas:
    print("Foram encontrados nomes de colunas duplicados:")
    for coluna in colunas_duplicadas:
        print(f"- {coluna}")
else:
    print("Nenhum nome de coluna duplicado identificado.")

# ------------------------------------------------------------
# 8. INSPEÇÃO DE VALORES AUSENTES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. VALORES AUSENTES")
print("=" * 70)

# Como a leitura preservou os textos, consideramos ausentes
# somente células vazias ou compostas exclusivamente por espaços.

df_para_inspecao = df.replace(r"^\s*$", pd.NA, regex=True)

ausentes = df_para_inspecao.isna().sum()
percentual_ausentes = df_para_inspecao.isna().mean().mul(100).round(2)

relatorio_ausentes = pd.DataFrame(
    {"valores_ausentes": ausentes, "percentual_ausentes": percentual_ausentes}
)

relatorio_ausentes = relatorio_ausentes[
    relatorio_ausentes["valores_ausentes"] > 0
].sort_values("valores_ausentes", ascending=False)

if relatorio_ausentes.empty:
    print("Nenhuma célula vazia identificada.")
else:
    print(relatorio_ausentes.to_string())

# ------------------------------------------------------------
# 9. LINHAS E COLUNAS TOTALMENTE VAZIAS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. ESTRUTURA DE VAZIOS")
print("=" * 70)

linhas_vazias = int(df_para_inspecao.isna().all(axis=1).sum())
colunas_vazias = int(df_para_inspecao.isna().all(axis=0).sum())

print(f"Linhas totalmente vazias: {linhas_vazias}")
print(f"Colunas totalmente vazias: {colunas_vazias}")

# ------------------------------------------------------------
# 10. VISUALIZAÇÃO DAS PRIMEIRAS LINHAS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. PRIMEIRAS CINCO LINHAS")
print("=" * 70)

display(df.head())

# ------------------------------------------------------------
# 11. RESUMO TÉCNICO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("9. RESUMO TÉCNICO")
print("=" * 70)

print(f"Arquivo analisado: {ARQUIVO_BASE.name}")
print(f"Registros: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print(f"Células: {df.size}")
print(f"Colunas duplicadas: {len(colunas_duplicadas)}")
print(f"Linhas totalmente vazias: {linhas_vazias}")
print(f"Colunas totalmente vazias: {colunas_vazias}")

print("\n" + "=" * 70)
print("INSPEÇÃO INICIAL CONCLUÍDA")
print("=" * 70)

print("Arquivo original preservado.")
print("Nenhum registro foi removido.")
print("Nenhuma coluna foi renomeada.")
print("Nenhuma transformação foi aplicada à base original.")
print("Próxima etapa: construir o dicionário de dados.")

UM OLHAR DE QUEM ESTÁ NO CAMPO
ETAPA 01 — INSPEÇÃO INICIAL

Diretório do projeto: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo
Diretório de dados: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data
Base original: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fonte_original
Dados derivados: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\derivados
Fontes públicas: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fontes_publicas

1. ESTRUTURA DE DIRETÓRIOS
EXISTE: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data
EXISTE: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fonte_original
EXISTE: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_qu

,Carimbo de data/hora,Estado,Gênero,Escolaridade,Faixa Etária,Área de Atuação,Tempo de atuação na Área,O que mais influenciou sua trajetória na área em que atua hoje? \n,O que mais define seu cotidiano profissional hoje?,Quanto sua realidade profissional corresponde ao que você imaginava quando começou?,O que você considera mais importante para quem está começando na sua área?,"Há algum alerta, tendência, recomendação ou outro ponto sobre sua experiência que você gostaria de registrar?","CONTATO — OPCIONAL \nQuer receber os resultados da pesquisa ou manter contato? Se desejar, deixe seu e-mail ou LinkedIn","Com que frequência você utiliza ferramentas de IA generativa (ex: ChatGPT, Copilot, Claude) no seu trabalho com dados?",Para quais dessas finalidades você já utilizou IA no seu trabalho? (marque todas que se aplicam),Como você ficou sabendo desta pesquisa?,Coluna 13
0,16/08/2026 20:03:18,Santa Catarina,Masculino,Técnico,Até 24,Desenvolvimento de Software,4–7 anos,"Formação acadêmica ou cursos, Indicação ou inf...","Resolução de problemas, Conhecimento técnico,...",3,"Desenvolver conhecimento técnico, Buscar exper...",,https://linkedin.com/in/luiz-vargas,,,,
1,16/08/2026 20:18:09,Santa Catarina,Feminino,Graduação,45–54,Arquitetura de Soluções,Mais de 15 anos,Formação acadêmica ou cursos,"Resolução de problemas, Conhecimento técnico,...",4,"Aprender continuamente, Desenvolver habilidade...",,mas.rosa74@gmail.com,,,,
2,16/08/2026 21:22:21,Santa Catarina,Feminino,Doutorado,45–54,"Desenvolvimento de Software, UX/ UI/ Design de...",1–3 anos,"Autoaprendizado e curiosidade pessoal, Transiç...","Resolução de problemas, Aprendizado contínuo,...",4,"Aprender continuamente, visão para a resolução...",,samyacampana@gmail.com,,,,
3,18/08/2026 11:26:10,Santa Catarina,Feminino,Pós-graduação,45–54,Produto/ Produto Management,8–15 anos,"Autoaprendizado e curiosidade pessoal, Acaso o...","Conhecimento técnico, Comunicação e colaboraçã...",3,"Construir uma rede de contatos, Conhecer difer...",,,,,,
4,18/08/2026 16:56:35,Santa Catarina,Masculino,Pós-graduação,55 ou mais,Cibersegurança,Mais de 15 anos,"Formação acadêmica ou cursos, Autoaprendizado ...","Resolução de problemas, Conhecimento técnico,...",5,"Desenvolver conhecimento técnico, Desenvolver ...",,,,,,



9. RESUMO TÉCNICO
Arquivo analisado: Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv
Registros: 54
Colunas: 17
Células: 918
Colunas duplicadas: 0
Linhas totalmente vazias: 0
Colunas totalmente vazias: 1

INSPEÇÃO INICIAL CONCLUÍDA
Arquivo original preservado.
Nenhum registro foi removido.
Nenhuma coluna foi renomeada.
Nenhuma transformação foi aplicada à base original.
Próxima etapa: construir o dicionário de dados.


In [9]:
# ============================================================
# ETAPA 02 — ESTRUTURA INICIAL DO DICIONÁRIO DE DADOS
# Projeto: Um Olhar de Quem Está no Campo
# ============================================================

from pathlib import Path
from datetime import datetime

import pandas as pd

# ------------------------------------------------------------
# 1. Definir o caminho de saída
# ------------------------------------------------------------

ARQUIVO_DICIONARIO = ROOT / "docs" / "analise" / "dicionario_de_dados.md"

# ------------------------------------------------------------
# 2. Validar a existência da base em memória
# ------------------------------------------------------------

if "df" not in globals():
    raise NameError(
        "O DataFrame 'df' não foi encontrado. "
        "Execute primeiro a célula que carrega a base original."
    )

# ------------------------------------------------------------
# 3. Registrar as características estruturais
# ------------------------------------------------------------

total_registros = len(df)
total_colunas = len(df.columns)

valores_ausentes = df.isna().sum()
percentual_ausentes = df.isna().mean().mul(100).round(2)

valores_unicos = df.nunique(dropna=True)

# ------------------------------------------------------------
# 4. Montar o documento
# ------------------------------------------------------------

linhas = [
    "# Dicionário de Dados",
    "",
    "## 1. Identificação",
    "",
    "- **Projeto:** Um Olhar de Quem Está no Campo",
    "- **Etapa:** Inspeção estrutural da base",
    "- **Arquivo de referência:** confirmar versão e origem",
    f"- **Registros identificados:** {total_registros}",
    f"- **Colunas identificadas:** {total_colunas}",
    f"- **Data de geração:** {datetime.now():%d/%m/%Y}",
    "",
    "## 2. Objetivo",
    "",
    "Documentar as variáveis da base de pesquisa, "
    "seus significados, tipos, categorias e condições "
    "de preenchimento, preservando a rastreabilidade "
    "das decisões analíticas.",
    "",
    "## 3. Dicionário de variáveis",
    "",
    "| Nº | Nome original da coluna | Tipo observado | "
    "Valores ausentes | % ausentes | Valores distintos | "
    "Significado da variável | Observações metodológicas |",
    "|---:|---|---|---:|---:|---:|---|---|",
]

for numero, coluna in enumerate(df.columns, start=1):

    tipo = str(df[coluna].dtype)
    ausentes = int(valores_ausentes[coluna])
    percentual = float(percentual_ausentes[coluna])
    distintos = int(valores_unicos[coluna])

    linhas.append(
        f"| {numero} "
        f"| `{coluna}` "
        f"| {tipo} "
        f"| {ausentes} "
        f"| {percentual:.2f}% "
        f"| {distintos} "
        f"| A documentar com base no questionário "
        f"| A verificar |"
    )

# ------------------------------------------------------------
# 5. Registrar cuidados metodológicos
# ------------------------------------------------------------

linhas.extend(
    [
        "",
        "## 4. Cuidados metodológicos",
        "",
        "- Os nomes originais das colunas foram preservados.",
        "- Nenhum registro foi removido nesta etapa.",
        "- Nenhuma transformação foi aplicada à base.",
        "- Valores ausentes não foram preenchidos.",
        "- Tipos observados pelo pandas não determinam, "
        "sozinhos, o significado metodológico das variáveis.",
        "- As perguntas adicionadas após o 36º registro "
        "devem ser identificadas conforme o questionário.",
        "- As variáveis adicionadas posteriormente possuem "
        "base efetiva de respostas menor que a base total.",
        "- A ausência de resposta a uma pergunta que ainda "
        "não existia no formulário não deve ser interpretada "
        "como falta de resposta do participante.",
        "",
        "## 5. Pendências para validação",
        "",
        "- Confirmar a versão e a origem do arquivo analisado.",
        "- Associar cada coluna à pergunta correspondente.",
        "- Identificar categorias e opções de resposta.",
        "- Diferenciar variáveis simples e múltipla escolha.",
        "- Verificar se há campos de texto livre.",
        "- Documentar a coluna totalmente vazia identificada " "na inspeção inicial.",
        "- Confirmar quais variáveis foram adicionadas após " "a resposta 36.",
        "- Validar a quantidade de respostas efetivas por variável.",
        "",
        "## 6. Critério de conclusão",
        "",
        "O dicionário será considerado validado quando cada "
        "variável tiver seu significado confirmado no "
        "instrumento de pesquisa e suas particularidades "
        "metodológicas documentadas.",
        "",
    ]
)

# ------------------------------------------------------------
# 6. Salvar o documento
# ------------------------------------------------------------

ARQUIVO_DICIONARIO.parent.mkdir(parents=True, exist_ok=True)

ARQUIVO_DICIONARIO.write_text("\n".join(linhas), encoding="utf-8")

print("DICIONÁRIO DE DADOS — ESTRUTURA INICIAL")
print("=" * 55)
print(f"Registros considerados: {total_registros}")
print(f"Colunas documentadas: {total_colunas}")
print(f"Arquivo criado: {ARQUIVO_DICIONARIO}")
print()
print("Estrutura gerada. Significados e observações")
print("metodológicas ainda precisam ser validados.")

DICIONÁRIO DE DADOS — ESTRUTURA INICIAL
Registros considerados: 54
Colunas documentadas: 17
Arquivo criado: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\docs\analise\dicionario_de_dados.md

Estrutura gerada. Significados e observações
metodológicas ainda precisam ser validados.


In [10]:
# ============================================================
# VERIFICAÇÃO 1 — NOMES E CONTEÚDO DAS COLUNAS
# ============================================================

# Exibir todos os nomes originais das colunas
print("NOMES ORIGINAIS DAS COLUNAS")
print("=" * 70)

for numero, coluna in enumerate(df.columns, start=1):
    print(f"{numero:02d}. {repr(coluna)}")


# Verificar a coluna suspeita
print("\n" + "=" * 70)
print("VERIFICAÇÃO DA COLUNA 17")
print("=" * 70)

coluna_suspeita = df.columns[16]

print(f"Nome: {repr(coluna_suspeita)}")
print(f"Valores não nulos: {df[coluna_suspeita].notna().sum()}")
print(f"Valores nulos: {df[coluna_suspeita].isna().sum()}")
print(f"Valores distintos: {df[coluna_suspeita].nunique(dropna=True)}")

print("\nDistribuição dos valores:")
display(df[coluna_suspeita].value_counts(dropna=False))


# Conferir o tamanho efetivo da base carregada
print("\n" + "=" * 70)
print("DIMENSÕES DA BASE")
print("=" * 70)

print(f"Registros: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

NOMES ORIGINAIS DAS COLUNAS
01. 'Carimbo de data/hora'
02. 'Estado'
03. 'Gênero'
04. 'Escolaridade'
05. 'Faixa Etária'
06. ' Área de Atuação'
07. ' Tempo de atuação na Área'
08. 'O que mais influenciou sua trajetória na área em que atua hoje? \n'
09. 'O que mais define seu cotidiano profissional hoje? '
10. 'Quanto sua realidade profissional corresponde ao que você imaginava quando começou? '
11. 'O que você considera mais importante para quem está começando na sua área? '
12. 'Há algum alerta, tendência, recomendação ou outro ponto sobre sua experiência que você gostaria de registrar?  '
13. 'CONTATO — OPCIONAL  \nQuer receber os resultados da pesquisa ou manter contato? Se desejar, deixe seu e-mail ou LinkedIn '
14. 'Com que frequência você utiliza ferramentas de IA generativa (ex: ChatGPT, Copilot, Claude) no seu trabalho com dados?  '
15. 'Para quais dessas finalidades você já utilizou IA no seu trabalho? (marque todas que se aplicam)'
16. 'Como você ficou sabendo desta pesquisa?  

Coluna 13
    54
Name: count, dtype: int64


DIMENSÕES DA BASE
Registros: 54
Colunas: 17


In [11]:
# ============================================================
# VERIFICAÇÃO 2 — INVESTIGAÇÃO DA COLUNA "Coluna 13"
# ============================================================

coluna_suspeita = "Coluna 13"

print("VALORES DA COLUNA")
print("=" * 60)
print(df[coluna_suspeita].value_counts(dropna=False))

print("\nEXEMPLOS DE REGISTROS")
print("=" * 60)

display(df[[df.columns[0], coluna_suspeita]].head(10))

print("\nVERIFICAÇÃO DE OUTRAS COLUNAS CONSTANTES")
print("=" * 60)

for coluna in df.columns:
    quantidade = df[coluna].nunique(dropna=False)

    if quantidade == 1:
        print(f"Coluna: {repr(coluna)} | " f"Valor: {df[coluna].iloc[0]!r}")

VALORES DA COLUNA
Coluna 13
    54
Name: count, dtype: int64

EXEMPLOS DE REGISTROS


,Carimbo de data/hora,Coluna 13
0,16/08/2026 20:03:18,
1,16/08/2026 20:18:09,
2,16/08/2026 21:22:21,
3,18/08/2026 11:26:10,
4,18/08/2026 16:56:35,
5,18/08/2026 17:01:51,
6,18/08/2026 18:38:16,
7,19/08/2026 09:55:01,
8,20/08/2026 10:56:32,
9,20/08/2026 11:42:20,



VERIFICAÇÃO DE OUTRAS COLUNAS CONSTANTES
Coluna: 'Coluna 13' | Valor: ''


| 17 | `Coluna 13` | str | 54 | 100.00% | 1 | Sem conteúdo informativo na base examinada | Todas as células contêm string vazia. Investigar a origem da coluna antes de decidir sobre sua exclusão de eventuais bases analíticas derivadas. |

In [12]:
# ============================================================
# VERIFICAÇÃO 3 — VALORES AUSENTES E CÉLULAS VAZIAS
# ============================================================

import pandas as pd

# Cópia apenas para inspeção: a base original não será alterada.
df_inspecao = df.copy()

# Identificar valores nulos, strings vazias e espaços em branco.
relatorio_ausentes = []

for coluna in df_inspecao.columns:
    serie = df_inspecao[coluna]

    nulos = serie.isna()
    strings_vazias = serie.map(lambda valor: isinstance(valor, str) and valor == "")
    somente_espacos = serie.map(
        lambda valor: isinstance(valor, str) and valor != "" and valor.strip() == ""
    )

    total_vazios = nulos | strings_vazias | somente_espacos

    relatorio_ausentes.append(
        {
            "coluna": coluna,
            "total_registros": len(serie),
            "valores_nulos": int(nulos.sum()),
            "strings_vazias": int(strings_vazias.sum()),
            "somente_espacos": int(somente_espacos.sum()),
            "total_sem_conteudo": int(total_vazios.sum()),
            "percentual_sem_conteudo": round(total_vazios.mean() * 100, 2),
        }
    )

relatorio_ausentes = pd.DataFrame(relatorio_ausentes)

display(
    relatorio_ausentes.sort_values(
        "percentual_sem_conteudo", ascending=False
    ).reset_index(drop=True)
)

print("\nRESUMO DA VERIFICAÇÃO")
print("=" * 60)
print(f"Registros examinados: {df_inspecao.shape[0]}")
print(f"Colunas examinadas: {df_inspecao.shape[1]}")
print("A base original permanece preservada.")

,coluna,total_registros,valores_nulos,strings_vazias,somente_espacos,total_sem_conteudo,percentual_sem_conteudo
0,Coluna 13,54,0,54,0,54,100.00
1,Como você ficou sabendo desta pesquisa?,54,0,36,0,36,66.67
2,Para quais dessas finalidades você já utilizou...,54,0,36,0,36,66.67
3,Com que frequência você utiliza ferramentas de...,54,0,36,0,36,66.67
4,"Há algum alerta, tendência, recomendação ou ou...",54,0,30,0,30,55.56
5,CONTATO — OPCIONAL \nQuer receber os resultad...,54,0,24,0,24,44.44
6,Carimbo de data/hora,54,0,0,0,0,0.00
7,Estado,54,0,0,0,0,0.00
8,Gênero,54,0,0,0,0,0.00
9,O que mais define seu cotidiano profissional h...,54,0,0,0,0,0.00



RESUMO DA VERIFICAÇÃO
Registros examinados: 54
Colunas examinadas: 17
A base original permanece preservada.


In [13]:
# ============================================================
# PRÓXIMA ETAPA — VALIDAR COLUNAS E AUSÊNCIAS
# Projeto: Um Olhar de Quem Está no Campo
# Objetivo: apoiar a validação do dicionário de dados
# ============================================================

print("=" * 70)
print("VALIDAÇÃO DAS COLUNAS E DAS AUSÊNCIAS")
print("=" * 70)

for numero, coluna in enumerate(df.columns, start=1):
    valores = df[coluna]

    # Considera células vazias ou compostas apenas por espaços
    vazios = valores.isna() | valores.astype("string").str.strip().eq("").fillna(False)

    quantidade_vazios = int(vazios.sum())
    percentual_vazios = round(quantidade_vazios / len(df) * 100, 2)

    print(f"\n{numero:02d}. {coluna}")
    print(f"    Registros sem conteúdo: {quantidade_vazios}/{len(df)}")
    print(f"    Percentual sem conteúdo: {percentual_vazios}%")

    if quantidade_vazios == len(df):
        print("    SITUAÇÃO: coluna completamente vazia.")
    elif quantidade_vazios > 0:
        print("    SITUAÇÃO: ausência parcial de conteúdo.")
    else:
        print("    SITUAÇÃO: todos os registros possuem conteúdo.")

print("\n" + "=" * 70)
print("VALIDAÇÃO CONCLUÍDA — BASE ORIGINAL NÃO ALTERADA")
print("=" * 70)

VALIDAÇÃO DAS COLUNAS E DAS AUSÊNCIAS

01. Carimbo de data/hora
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúdo.

02. Estado
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúdo.

03. Gênero
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúdo.

04. Escolaridade
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúdo.

05. Faixa Etária
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúdo.

06.  Área de Atuação
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúdo.

07.  Tempo de atuação na Área
    Registros sem conteúdo: 0/54
    Percentual sem conteúdo: 0.0%
    SITUAÇÃO: todos os registros possuem conteúd

In [14]:
# ============================================================
# CONFERÊNCIA DAS COLUNAS COM AUSÊNCIAS
# Projeto: Um Olhar de Quem Está no Campo
# ============================================================

print("COLUNAS QUE PRECISAM DE ATENÇÃO")
print("=" * 70)

for coluna in df.columns:
    valores = df[coluna].astype("string").str.strip()

    sem_conteudo = df[coluna].isna() | valores.eq("").fillna(False)

    quantidade = int(sem_conteudo.sum())

    if quantidade > 0:
        percentual = round(quantidade / len(df) * 100, 2)

        print(f"\nColuna: {coluna}")
        print(f"Registros sem conteúdo: {quantidade}")
        print(f"Percentual: {percentual}%")

print("\n" + "=" * 70)
print("CONFERÊNCIA CONCLUÍDA")

COLUNAS QUE PRECISAM DE ATENÇÃO

Coluna: Há algum alerta, tendência, recomendação ou outro ponto sobre sua experiência que você gostaria de registrar?  
Registros sem conteúdo: 30
Percentual: 55.56%

Coluna: CONTATO — OPCIONAL  
Quer receber os resultados da pesquisa ou manter contato? Se desejar, deixe seu e-mail ou LinkedIn 
Registros sem conteúdo: 24
Percentual: 44.44%

Coluna: Com que frequência você utiliza ferramentas de IA generativa (ex: ChatGPT, Copilot, Claude) no seu trabalho com dados?  
Registros sem conteúdo: 36
Percentual: 66.67%

Coluna: Para quais dessas finalidades você já utilizou IA no seu trabalho? (marque todas que se aplicam)
Registros sem conteúdo: 36
Percentual: 66.67%

Coluna: Como você ficou sabendo desta pesquisa?  
Registros sem conteúdo: 36
Percentual: 66.67%

Coluna: Coluna 13
Registros sem conteúdo: 54
Percentual: 100.0%

CONFERÊNCIA CONCLUÍDA


In [16]:
from pathlib import Path

raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

print("Raiz do projeto:", raiz_projeto)
print("\nArquivos CSV encontrados:")

for arquivo in raiz_projeto.rglob("*.csv"):
    if ".venv" not in arquivo.parts:
        print(arquivo)

Raiz do projeto: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo

Arquivos CSV encontrados:
c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fonte_original\Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv


In [17]:
arquivo_original = (
    raiz_projeto
    / "data"
    / "fonte_original"
    / "Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv"
)

In [18]:
# ============================================================
# ETAPA 2 — PREPARAÇÃO DA BASE ANALÍTICA
# Projeto: Um Olhar de Quem Está no Campo
# Objetivo: criar uma cópia de trabalho sem alterar a base original
# ============================================================

from pathlib import Path
import pandas as pd

# 1. IDENTIFICAR A RAIZ DO PROJETO
raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

print("Raiz do projeto:", raiz_projeto)


# 2. LOCALIZAR O ARQUIVO ORIGINAL
arquivo_original = (
    raiz_projeto
    / "data"
    / "fonte_original"
    / "Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv"
)

# Verificar se o arquivo existe antes de continuar
if not arquivo_original.exists():
    raise FileNotFoundError(f"Arquivo original não encontrado:\n{arquivo_original}")

print("\nArquivo original localizado:")
print(arquivo_original)


# 3. DEFINIR O DESTINO DA BASE DERIVADA
pasta_derivados = raiz_projeto / "data" / "derivados"
pasta_derivados.mkdir(parents=True, exist_ok=True)

arquivo_derivado = pasta_derivados / "base_analitica_inicial.csv"


# 4. LER A BASE ORIGINAL
# Preservar os textos como strings e manter células vazias
# como strings vazias nesta primeira leitura.
df_original = pd.read_csv(arquivo_original, dtype="string", keep_default_na=False)

print("\nBASE ORIGINAL")
print("Quantidade de linhas:", df_original.shape[0])
print("Quantidade de colunas:", df_original.shape[1])


# 5. CRIAR UMA CÓPIA DE TRABALHO
# Nenhuma transformação será aplicada ao arquivo original.
df_analise = df_original.copy()


# 6. PADRONIZAR OS NOMES DAS COLUNAS
# Remover espaços excedentes no início e no fim dos cabeçalhos.
df_analise.columns = df_analise.columns.str.strip()

# Verificar se a padronização gerou nomes duplicados.
colunas_duplicadas = df_analise.columns[df_analise.columns.duplicated()].tolist()

if colunas_duplicadas:
    raise ValueError(
        "A padronização dos cabeçalhos gerou nomes duplicados: " f"{colunas_duplicadas}"
    )


# 7. CONVERTER CÉLULAS VAZIAS EM VALORES AUSENTES
# Isso permite identificar corretamente os campos sem resposta.
df_analise = df_analise.replace(r"^\s*$", pd.NA, regex=True)


# 8. CONFERIR A ESTRUTURA DA BASE ANALÍTICA
print("\nBASE ANALÍTICA")
print("Quantidade de linhas:", df_analise.shape[0])
print("Quantidade de colunas:", df_analise.shape[1])
print("Dimensões:", df_analise.shape)

print("\nValores ausentes por coluna:")
print(df_analise.isna().sum())


# 9. IDENTIFICAR COLUNAS TOTALMENTE VAZIAS
colunas_vazias = [
    coluna for coluna in df_analise.columns if df_analise[coluna].isna().all()
]

print("\nColunas totalmente vazias:")
print(colunas_vazias if colunas_vazias else "Nenhuma")


# 10. SALVAR A BASE ANALÍTICA INICIAL
# O arquivo original permanece preservado.
df_analise.to_csv(arquivo_derivado, index=False, encoding="utf-8-sig")

print("\nPREPARAÇÃO CONCLUÍDA")
print("Arquivo derivado salvo em:")
print(arquivo_derivado)

Raiz do projeto: c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo

Arquivo original localizado:
c:\Users\User\OneDrive\Desktop\Portifolio_Data_Analytics\06_Um_olhar_de_quem_esta_no_campo\data\fonte_original\Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv

BASE ORIGINAL
Quantidade de linhas: 54
Quantidade de colunas: 17

BASE ANALÍTICA
Quantidade de linhas: 54
Quantidade de colunas: 17
Dimensões: (54, 17)

Valores ausentes por coluna:
Carimbo de data/hora                                                                                                         0
Estado                                                                                                                       0
Gênero                                                                                                                       0
Escolaridade                                                                                                         

## Registro 02 — Preparação da base analítica inicial

**Data:** 27/10/2026

### Objetivo
Criar uma cópia de trabalho da base original para as análises,
preservando o arquivo de origem.

### Procedimentos realizados
- Leitura do CSV original com pandas.
- Criação de uma cópia independente para análise.
- Remoção de espaços excedentes no início e no fim dos cabeçalhos.
- Verificação de nomes de colunas duplicados após a padronização.
- Conversão de células vazias ou compostas apenas por espaços
  em valores ausentes (`pd.NA`).
- Inspeção das dimensões e dos valores ausentes.
- Salvamento da base derivada em
  `data/derivados/base_analitica_inicial.csv`.

### Resultado
- Base original: 54 linhas e 17 colunas.
- Base analítica inicial: 54 linhas e 17 colunas.
- Nenhuma linha foi excluída nesta etapa.
- Nenhuma coluna foi excluída nesta etapa.
- O arquivo original não foi sobrescrito.

### Limitações e cuidados
As ausências nas perguntas adicionadas após a 36ª resposta
devem ser tratadas como ausências estruturais.

A coluna `Coluna 13` permanece na base até a conclusão
da verificação documental e da decisão sobre sua exclusão
da versão analítica.

### Próxima etapa
Validar a distribuição dos valores ausentes e conferir a
integridade da base derivada antes de iniciar a análise descritiva.

In [19]:
# ============================================================
# ETAPA 3 — VALIDAÇÃO DA BASE ANALÍTICA
# Projeto: Um Olhar de Quem Está no Campo
# ============================================================

from pathlib import Path
import pandas as pd

# 1. Localizar a raiz do projeto
raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# 2. Definir os caminhos dos arquivos
arquivo_original = (
    raiz_projeto
    / "data"
    / "fonte_original"
    / "Pesquisa — Um Olhar de Quem Está no Campo - BASELINE_36_RESPOSTAS.csv"
)

arquivo_derivado = raiz_projeto / "data" / "derivados" / "base_analitica_inicial.csv"

# 3. Verificar a existência dos arquivos
assert arquivo_original.exists(), "Arquivo original não encontrado."
assert arquivo_derivado.exists(), "Arquivo derivado não encontrado."

# 4. Ler os arquivos para comparação
df_original = pd.read_csv(arquivo_original, dtype="string", keep_default_na=False)

df_analise = pd.read_csv(arquivo_derivado, dtype="string", keep_default_na=False)

# 5. Comparar dimensões
print("1. DIMENSÕES")
print(f"Original: {df_original.shape}")
print(f"Derivada: {df_analise.shape}")

# 6. Comparar quantidade de respostas
print("\n2. QUANTIDADE DE REGISTROS")
print(f"Original: {len(df_original)}")
print(f"Derivada: {len(df_analise)}")

# 7. Conferir os cabeçalhos
print("\n3. CABEÇALHOS")
print("Quantidade de colunas originais:", len(df_original.columns))
print("Quantidade de colunas derivadas:", len(df_analise.columns))
print("Cabeçalhos derivados duplicados:", df_analise.columns.duplicated().sum())

# 8. Calcular valores ausentes na base derivada
# O arquivo CSV representa valores ausentes como campos vazios.
df_validacao = df_analise.replace(r"^\s*$", pd.NA, regex=True)

resumo_ausencias = pd.DataFrame(
    {
        "coluna": df_validacao.columns,
        "ausentes": df_validacao.isna().sum().values,
        "percentual_ausente": (df_validacao.isna().mean().values * 100).round(2),
    }
)

resumo_ausencias = resumo_ausencias.sort_values(by="ausentes", ascending=False)

print("\n4. RESUMO DE AUSÊNCIAS")
display(resumo_ausencias)

# 9. Verificar colunas completamente vazias
colunas_vazias = [
    coluna for coluna in df_validacao.columns if df_validacao[coluna].isna().all()
]

print("\n5. COLUNAS COMPLETAMENTE VAZIAS")
print(colunas_vazias if colunas_vazias else "Nenhuma")

# 10. Verificar se o número de registros foi preservado
print("\n6. INTEGRIDADE DOS REGISTROS")

if len(df_original) == len(df_analise):
    print("OK: a quantidade de registros foi preservada.")
else:
    print("ATENÇÃO: a quantidade de registros mudou.")

# 11. Verificar se o conteúdo das respostas foi preservado,
# desconsiderando apenas os espaços externos dos cabeçalhos.
original_com_cabecalhos_padronizados = df_original.copy()
original_com_cabecalhos_padronizados.columns = (
    original_com_cabecalhos_padronizados.columns.str.strip()
)

original_com_cabecalhos_padronizados = original_com_cabecalhos_padronizados.replace(
    r"^\s*$", pd.NA, regex=True
)

conteudo_preservado = original_com_cabecalhos_padronizados.equals(df_validacao)

print(
    "Conteúdo das respostas preservado:",
    "Sim" if conteudo_preservado else "Não — investigar diferenças.",
)

1. DIMENSÕES
Original: (54, 17)
Derivada: (54, 17)

2. QUANTIDADE DE REGISTROS
Original: 54
Derivada: 54

3. CABEÇALHOS
Quantidade de colunas originais: 17
Quantidade de colunas derivadas: 17
Cabeçalhos derivados duplicados: 0

4. RESUMO DE AUSÊNCIAS


,coluna,ausentes,percentual_ausente
16,Coluna 13,54,100.00
15,Como você ficou sabendo desta pesquisa?,36,66.67
14,Para quais dessas finalidades você já utilizou...,36,66.67
13,Com que frequência você utiliza ferramentas de...,36,66.67
11,"Há algum alerta, tendência, recomendação ou ou...",30,55.56
12,CONTATO — OPCIONAL \nQuer receber os resultad...,24,44.44
0,Carimbo de data/hora,0,0.00
1,Estado,0,0.00
2,Gênero,0,0.00
8,O que mais define seu cotidiano profissional h...,0,0.00



5. COLUNAS COMPLETAMENTE VAZIAS
['Coluna 13']

6. INTEGRIDADE DOS REGISTROS
OK: a quantidade de registros foi preservada.
Conteúdo das respostas preservado: Sim


In [20]:
# ============================================================
# ETAPA 4 — BASE ANALÍTICA VERSÃO 1
# Projeto: Um Olhar de Quem Está no Campo
# ============================================================

from pathlib import Path
import pandas as pd

# 1. Localizar a raiz do projeto
raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# 2. Definir os caminhos
arquivo_entrada = raiz_projeto / "data" / "derivados" / "base_analitica_inicial.csv"

arquivo_saida = raiz_projeto / "data" / "derivados" / "base_analitica_v1.csv"

# 3. Carregar a base inicial
df_v1 = pd.read_csv(arquivo_entrada, dtype="string", keep_default_na=False)

# 4. Converter campos vazios em valores ausentes
df_v1 = df_v1.replace(r"^\s*$", pd.NA, regex=True)

# 5. Registrar as dimensões anteriores
dimensoes_antes = df_v1.shape

# 6. Excluir somente a coluna totalmente vazia
coluna_excluida = "Coluna 13"

if coluna_excluida not in df_v1.columns:
    raise KeyError(f"A coluna '{coluna_excluida}' não foi encontrada.")

if not df_v1[coluna_excluida].isna().all():
    raise ValueError(
        f"A coluna '{coluna_excluida}' contém dados. " "A exclusão foi cancelada."
    )

df_v1 = df_v1.drop(columns=[coluna_excluida])

# 7. Validar o resultado
dimensoes_depois = df_v1.shape

print("DIMENSÕES")
print("Antes:", dimensoes_antes)
print("Depois:", dimensoes_depois)

print("\nCOLUNAS EXCLUÍDAS")
print([coluna_excluida])

print("\nVALORES AUSENTES POR COLUNA")
print(df_v1.isna().sum())

# 8. Salvar a nova versão
df_v1.to_csv(arquivo_saida, index=False, encoding="utf-8-sig")

print("\nBASE ANALÍTICA V1 CRIADA")
print("Arquivo:", arquivo_saida)

DIMENSÕES
Antes: (54, 17)
Depois: (54, 16)

COLUNAS EXCLUÍDAS
['Coluna 13']

VALORES AUSENTES POR COLUNA
Carimbo de data/hora                                                                                                         0
Estado                                                                                                                       0
Gênero                                                                                                                       0
Escolaridade                                                                                                                 0
Faixa Etária                                                                                                                 0
Área de Atuação                                                                                                              0
Tempo de atuação na Área                                                                                                     0
O que 

In [21]:
# ============================================================
# ETAPA 5 — ANÁLISE DESCRITIVA DO PERFIL
# Projeto: Um Olhar de Quem Está no Campo
# ============================================================

from pathlib import Path
import pandas as pd

# 1. Localizar a raiz do projeto
raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# 2. Carregar a base analítica V1
arquivo_v1 = raiz_projeto / "data" / "derivados" / "base_analitica_v1.csv"

df_v1 = pd.read_csv(arquivo_v1, dtype="string", keep_default_na=False)

# 3. Definir as variáveis de perfil
variaveis_perfil = ["Estado", "Gênero", "Escolaridade", "Faixa Etária"]

# 4. Verificar se todas as colunas existem
colunas_inexistentes = [
    coluna for coluna in variaveis_perfil if coluna not in df_v1.columns
]

if colunas_inexistentes:
    raise KeyError(f"Colunas não encontradas: {colunas_inexistentes}")

# 5. Gerar tabelas de frequência
for coluna in variaveis_perfil:

    print("\n" + "=" * 60)
    print(f"VARIÁVEL: {coluna}")
    print("=" * 60)

    # Identificar respostas vazias
    respostas = df_v1[coluna].str.strip()
    respostas_validas = respostas[respostas.ne("")]

    frequencia = respostas_validas.value_counts()

    tabela = pd.DataFrame(
        {
            "Frequência": frequencia,
            "Percentual válido (%)": (frequencia / len(respostas_validas) * 100).round(
                2
            ),
        }
    )

    print(f"Total de participantes: {len(df_v1)}")
    print(f"Respostas válidas: {len(respostas_validas)}")
    print(f"Respostas vazias: {len(df_v1) - len(respostas_validas)}")

    display(tabela)


VARIÁVEL: Estado
Total de participantes: 54
Respostas válidas: 54
Respostas vazias: 0


,Frequência,Percentual válido (%)
Estado,,
Santa Catarina,46,85.19
Rio Grande do Sul,2,3.7
São Paulo,1,1.85
Espírito Santo,1,1.85
Piauí,1,1.85
Distrito Federal,1,1.85
Rio de Janeiro,1,1.85
Minas Gerais,1,1.85



VARIÁVEL: Gênero
Total de participantes: 54
Respostas válidas: 54
Respostas vazias: 0


,Frequência,Percentual válido (%)
Gênero,,
Masculino,40,74.07
Feminino,13,24.07
Prefiro não informar,1,1.85



VARIÁVEL: Escolaridade
Total de participantes: 54
Respostas válidas: 54
Respostas vazias: 0


,Frequência,Percentual válido (%)
Escolaridade,,
Graduação,22,40.74
Pós-graduação,17,31.48
Técnico,6,11.11
Doutorado,4,7.41
Ensino médio,3,5.56
Mestrado,2,3.7



VARIÁVEL: Faixa Etária
Total de participantes: 54
Respostas válidas: 54
Respostas vazias: 0


,Frequência,Percentual válido (%)
Faixa Etária,,
45–54,14,25.93
35–44,14,25.93
Até 24,9,16.67
25–34,9,16.67
55 ou mais,7,12.96
Prefiro não informar,1,1.85


In [22]:
# ============================================================
# ETAPA 6 — INVESTIGAÇÃO DA TRAJETÓRIA PROFISSIONAL
# Projeto: Um Olhar de Quem Está no Campo
# ============================================================

from pathlib import Path
import pandas as pd

# 1. Localizar a raiz do projeto
raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# 2. Carregar a base analítica V1
arquivo_v1 = raiz_projeto / "data" / "derivados" / "base_analitica_v1.csv"

df_v1 = pd.read_csv(arquivo_v1, dtype="string", keep_default_na=False)

# 3. Definir a variável de interesse
variavel = "O que mais influenciou sua trajetória na área em que atua hoje?"

# 4. Conferir se a coluna existe
if variavel not in df_v1.columns:
    print("Coluna não encontrada.")
    print("\nColunas disponíveis:")

    for coluna in df_v1.columns:
        print(repr(coluna))

else:
    # 5. Identificar respostas preenchidas
    respostas = df_v1[variavel].str.strip()
    respostas_validas = respostas[respostas.ne("")]

    print("INVESTIGAÇÃO DA VARIÁVEL")
    print("-" * 60)

    print("Total de participantes:", len(df_v1))
    print("Respostas preenchidas:", len(respostas_validas))
    print("Respostas vazias:", len(df_v1) - len(respostas_validas))

    # 6. Examinar as respostas exatamente como estão na base
    print("\nRESPOSTAS DISTINTAS E SUAS FREQUÊNCIAS")

    frequencias = respostas_validas.value_counts()

    tabela_inicial = pd.DataFrame(
        {
            "Frequência": frequencias,
            "Percentual dos participantes (%)": (frequencias / len(df_v1) * 100).round(
                2
            ),
        }
    )

    display(tabela_inicial)

    # 7. Mostrar exemplos para entender a estrutura
    print("\nEXEMPLOS DE RESPOSTAS REGISTRADAS")

    for resposta in respostas_validas.drop_duplicates().head(10):
        print("-", resposta)

INVESTIGAÇÃO DA VARIÁVEL
------------------------------------------------------------
Total de participantes: 54
Respostas preenchidas: 54
Respostas vazias: 0

RESPOSTAS DISTINTAS E SUAS FREQUÊNCIAS


,Frequência,Percentual dos participantes (%)
O que mais influenciou sua trajetória na área em que atua hoje?,,
"Autoaprendizado e curiosidade pessoal, Transição de outra carreira, Busca por estabilidade ou oportunidade de mercado",6,11.11
"Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Transição de outra carreira",6,11.11
"Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal",6,11.11
"Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Acaso ou oportunidade inesperada",4,7.41
"Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Busca por estabilidade ou oportunidade de mercado",4,7.41
Transição de outra carreira,4,7.41
"Formação acadêmica ou cursos, Transição de outra carreira",4,7.41
"Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Indicação ou influência de rede de contatos",4,7.41
Formação acadêmica ou cursos,3,5.56



EXEMPLOS DE RESPOSTAS REGISTRADAS
- Formação acadêmica ou cursos, Indicação ou influência de rede de contatos
- Formação acadêmica ou cursos
- Autoaprendizado e curiosidade pessoal, Transição de outra carreira, Busca por estabilidade ou oportunidade de mercado
- Autoaprendizado e curiosidade pessoal, Acaso ou oportunidade inesperada
- Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Transição de outra carreira
- Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Acaso ou oportunidade inesperada
- Formação acadêmica ou cursos, Autoaprendizado e curiosidade pessoal, Busca por estabilidade ou oportunidade de mercado
- Transição de outra carreira
- Autoaprendizado e curiosidade pessoal, Transição de outra carreira
- Busca por estabilidade ou oportunidade de mercado


In [23]:
# ============================================================
# ETAPA 6.1 — FREQUÊNCIA DAS INFLUÊNCIAS NA TRAJETÓRIA
# ============================================================

import pandas as pd
from pathlib import Path

# 1. Localizar a raiz do projeto
raiz_projeto = Path.cwd()

if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# 2. Carregar a base analítica V1
arquivo_v1 = raiz_projeto / "data" / "derivados" / "base_analitica_v1.csv"

df_v1 = pd.read_csv(arquivo_v1, dtype="string", keep_default_na=False)

# 3. Selecionar a pergunta
variavel = "O que mais influenciou sua trajetória na área em que atua hoje?"

respostas = df_v1[variavel].str.strip()
respostas = respostas[respostas.ne("")]

# 4. Separar as alternativas registradas em cada resposta
influencias = respostas.str.split(", ").explode().str.strip()

# 5. Contar cada alternativa
frequencias = influencias.value_counts()

# 6. Construir a tabela de resultados
tabela_influencias = pd.DataFrame(
    {
        "Participantes que mencionaram": frequencias,
        "Percentual dos participantes (%)": (frequencias / len(df_v1) * 100).round(2),
    }
)

print("TOTAL DE PARTICIPANTES:", len(df_v1))
print("RESPOSTAS PREENCHIDAS:", len(respostas))
print("\nFREQUÊNCIA POR INFLUÊNCIA")

display(tabela_influencias)

TOTAL DE PARTICIPANTES: 54
RESPOSTAS PREENCHIDAS: 54

FREQUÊNCIA POR INFLUÊNCIA


,Participantes que mencionaram,Percentual dos participantes (%)
O que mais influenciou sua trajetória na área em que atua hoje?,,
Autoaprendizado e curiosidade pessoal,38,70.37
Formação acadêmica ou cursos,34,62.96
Transição de outra carreira,22,40.74
Busca por estabilidade ou oportunidade de mercado,13,24.07
Indicação ou influência de rede de contatos,7,12.96
Acaso ou oportunidade inesperada,7,12.96
Familiares da área,1,1.85
